# M1 v04 单细胞状态与响应轨迹

## 后期作图参数说明

本图只读取 M1 v04 正式 `A1_ACTIVE, dt=0.02` 保留的接受轨迹、表面网格和首个失败快照。A 面板显示四个已接受细胞状态，B 面板显示激活和整体形变读数；灰色区不是模拟结果。

In [1]:
# 作图调整参数：用户此前授权自行排版，长时间序列采用 10 × 5 in 主轴框
from pathlib import Path
from IPython.display import Image, Markdown, display

PREFIX = "FigM1_v04_cell_state_progress_v01_20260809"
REVISION_DIR = Path.cwd().resolve()
STYLE_SOURCE = "cb-plot-unified-style / unified-style-v01"
FIGURE_SIZE_IN = (10.0, 5.0)  # 版本包校验字段；表示主轴框，不是整张画布
AXIS_BOX_SIZE_IN = FIGURE_SIZE_IN
DPI = 600
PRIMARY_DATA_NAME = "01_FigM1_v04_cell_state_progress_v01_20260809_data.npy"
PRIMARY_DATA = REVISION_DIR / PRIMARY_DATA_NAME
OUTPUT_PNG = REVISION_DIR / f"04_{PREFIX}.png"
OUTPUT_SVG = REVISION_DIR / f"05_{PREFIX}.svg"

print(f"当前版本目录: {REVISION_DIR}")

当前版本目录: E:\Temp-Projects\PRL\results\route_h\stage2_gate_a_v04\Figures\FigM1_v04_cell_state_progress\FigM1_v04_cell_state_progress_v01_20260809


## 数据读取与处理

载入保留的二进制轨迹和 CSV。检查时间节点与顶点帧一一对应；不插值、不补齐失败后的时间区间。

In [2]:
import csv
import json
import numpy as np

if not PRIMARY_DATA.is_file():
    raise FileNotFoundError(f"缺少主数据文件: {PRIMARY_DATA}")
vertices = np.load(PRIMARY_DATA, allow_pickle=False)
faces = np.load(REVISION_DIR / f"01a_{PREFIX}_data_faces.npy", allow_pickle=False)
with (REVISION_DIR / f"01d_{PREFIX}_data_time_series.csv").open(encoding="utf-8", newline="") as handle:
    rows = list(csv.DictReader(handle))
failure = json.loads((REVISION_DIR / f"01b_{PREFIX}_data_failure.json").read_text(encoding="utf-8"))
assert vertices.shape[0] == len(rows)
assert faces.ndim == 2 and faces.shape[1] == 3
print({"accepted_nodes": len(rows), "last_time": rows[-1]["time"], "failure_time": failure["time"]})

{'accepted_nodes': 194, 'last_time': '3.8599999999999999', 'failure_time': 3.88}


## 绘图与导出

四个快照使用同一相机、空间尺度和位移色标。位移定义为相对初始顶点的欧氏距离除以初始纤维长度；形变曲线直接读取正式 time series。

In [3]:
import importlib.util

renderer_path = REVISION_DIR / f"03d_{PREFIX}_render.py"
spec = importlib.util.spec_from_file_location(f"{PREFIX}_render", renderer_path)
if spec is None or spec.loader is None:
    raise ImportError(f"无法加载绘图脚本: {renderer_path}")
renderer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(renderer)
render_metrics = renderer.render(REVISION_DIR, PREFIX)
print(render_metrics)

{'last_accepted_time': 3.86, 'failure_time': 3.88, 'failure_residual': 1.1991047754392014e-08, 'maximum_absolute_volume_error_percent': 0.015860840774739593, 'peak_axial_shortening_percent': 9.78085751907355, 'png': 'E:\\Temp-Projects\\PRL\\results\\route_h\\stage2_gate_a_v04\\Figures\\FigM1_v04_cell_state_progress\\FigM1_v04_cell_state_progress_v01_20260809\\04_FigM1_v04_cell_state_progress_v01_20260809.png', 'svg': 'E:\\Temp-Projects\\PRL\\results\\route_h\\stage2_gate_a_v04\\Figures\\FigM1_v04_cell_state_progress\\FigM1_v04_cell_state_progress_v01_20260809\\05_FigM1_v04_cell_state_progress_v01_20260809.svg', 'style_manifest': 'E:\\Temp-Projects\\PRL\\results\\route_h\\stage2_gate_a_v04\\Figures\\FigM1_v04_cell_state_progress\\FigM1_v04_cell_state_progress_v01_20260809\\04_FigM1_v04_cell_state_progress_v01_20260809_style_manifest.json'}


## 最终导出预览

In [4]:
if not OUTPUT_PNG.is_file() or OUTPUT_PNG.stat().st_size == 0:
    raise FileNotFoundError(f"缺少有效 PNG 导出: {OUTPUT_PNG}")
if not OUTPUT_SVG.is_file() or OUTPUT_SVG.stat().st_size == 0:
    raise FileNotFoundError(f"缺少有效 SVG 导出: {OUTPUT_SVG}")
display(Markdown(f"### 当前预览：{OUTPUT_PNG.name}"))
display(Image(filename=str(OUTPUT_PNG)))

### 当前预览：04_FigM1_v04_cell_state_progress_v01_20260809.png